# 03 — Proposed LCAD-AE, Ablation, and Cross-Domain Transfer

This notebook evaluates the lightweight proposed model in three ways:

1. full-model evaluation in both domains;
2. component ablation for reconstruction, statistics, and correlation terms;
3. cross-domain temporal-encoder transfer with limited target normal data.

The transfer experiment uses domain-specific input/output projections and transfers only the channel-independent temporal blocks.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from drift_detection.config import load_config

config = load_config()
print(f"Project root: {PROJECT_ROOT}")
print(f"Experiment version: {config['project']['experiment_version']}")
print(f"Scenario profile: {config['experiment']['profile']}")

from drift_detection.experiments_v2 import (
    manuscript_result_root,
    run_cross_domain_transfer_suite,
    run_proposed_manuscript_suite,
)


Project root: /home/commu10/Documents/BT/cross_domain_drift_detection_manuscript_v2/cross_domain_drift_detection
Experiment version: manuscript_v2
Scenario profile: manuscript


## Controls


In [2]:
SEEDS = list(config["project"]["seed_list"])
FORCE_RERUN = False
RUN_ABLATIONS = True
RUN_TRANSFER = True

ABLATION_VARIANTS = ["full", "recon_only", "recon_statistics", "recon_correlation"] if RUN_ABLATIONS else ["full"]
TRANSFER_FRACTIONS = list(config["proposed"]["transfer"]["target_normal_fractions"])
TRANSFER_DIRECTIONS = list(config["proposed"]["transfer"]["directions"])

print("Independent seeds:", SEEDS)
print("Proposed variants:", ABLATION_VARIANTS)
print("Transfer fractions:", TRANSFER_FRACTIONS)
print("Transfer directions:", TRANSFER_DIRECTIONS)


Independent seeds: [42, 52, 62, 72, 82]
Proposed variants: ['full', 'recon_only', 'recon_statistics', 'recon_correlation']
Transfer fractions: [0.01, 0.05, 0.1, 0.25, 1.0]
Transfer directions: ['gas_to_skab', 'skab_to_gas']


## Gas Sensor proposed-model experiments


In [3]:
gas_proposed = run_proposed_manuscript_suite(
    config["paths"]["benchmarks"] / "gas_controlled.npz",
    dataset="gas_sensor",
    config=config,
    seeds=SEEDS,
    variants=ABLATION_VARIANTS,
    force=FORCE_RERUN,
)
display(gas_proposed[[column for column in ["variant", "seed", "status", "f1", "auprc", "event_recall", "false_positive_rate", "topk_channel_recall"] if column in gas_proposed.columns]])


2026-07-29 08:14:14,287 | INFO | RUN_STARTED | dataset=gas_sensor variant=full seed=42
/home/commu10/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


LCAD-AE epoch 1/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 2/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 3/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 4/30:   0%|          | 0/4 [00:00<?, ?it/s]

...

LCAD-AE epoch 28/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 29/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 30/30:   0%|          | 0/4 [00:00<?, ?it/s]

2026-07-29 08:28:00,599 | INFO | RUN_FINISHED | status=completed


,variant,seed,status,f1,auprc,event_recall,false_positive_rate,topk_channel_recall
0,full,42,completed,0.134945,0.187184,0.496528,0.126655,0.569273
1,recon_only,42,completed,0.095699,0.160647,0.381944,0.126753,0.528475
2,recon_statistics,42,completed,0.134053,0.186991,0.496528,0.126655,0.568502
3,recon_correlation,42,completed,0.100077,0.161096,0.395833,0.126753,0.529673
4,full,52,completed,0.158582,0.185658,0.607639,0.144091,0.566688
5,recon_only,52,completed,0.102508,0.162058,0.409722,0.133870,0.522237
6,recon_statistics,52,completed,0.155143,0.185287,0.590278,0.144091,0.566372
7,recon_correlation,52,completed,0.105850,0.162691,0.413194,0.133870,0.523019
8,full,62,completed,0.156223,0.189860,0.586806,0.133268,0.577070
9,recon_only,62,completed,0.124785,0.162681,0.524306,0.146961,0.519728


## SKAB proposed-model experiments


In [4]:
skab_proposed = run_proposed_manuscript_suite(
    config["paths"]["benchmarks"] / "skab_controlled.npz",
    dataset="skab",
    config=config,
    seeds=SEEDS,
    variants=ABLATION_VARIANTS,
    force=FORCE_RERUN,
)
display(skab_proposed[[column for column in ["variant", "seed", "status", "f1", "auprc", "event_recall", "false_positive_rate", "topk_channel_recall", "native_f1"] if column in skab_proposed.columns]])


2026-07-29 08:28:00,685 | INFO | RUN_STARTED | dataset=skab variant=full seed=42


LCAD-AE epoch 1/30:   0%|          | 0/11 [00:00<?, ?it/s]

LCAD-AE epoch 2/30:   0%|          | 0/11 [00:00<?, ?it/s]

LCAD-AE epoch 3/30:   0%|          | 0/11 [00:00<?, ?it/s]

LCAD-AE epoch 4/30:   0%|          | 0/11 [00:00<?, ?it/s]

...

LCAD-AE epoch 28/30:   0%|          | 0/11 [00:00<?, ?it/s]

LCAD-AE epoch 29/30:   0%|          | 0/11 [00:00<?, ?it/s]

LCAD-AE epoch 30/30:   0%|          | 0/11 [00:00<?, ?it/s]

2026-07-29 08:43:51,762 | INFO | RUN_FINISHED | status=completed


,variant,seed,status,f1,auprc,event_recall,false_positive_rate,topk_channel_recall,native_f1
0,full,42,completed,0.251764,0.557719,0.975694,0.372341,0.696837,0.557721
1,recon_only,42,completed,0.161219,0.415547,0.968750,0.797772,0.492102,0.554965
2,recon_statistics,42,completed,0.245628,0.552432,0.972222,0.393943,0.689579,0.557474
3,recon_correlation,42,completed,0.193474,0.436079,0.975694,0.493785,0.495820,0.559136
4,full,52,completed,0.246471,0.560571,0.972222,0.376279,0.669197,0.558034
5,recon_only,52,completed,0.154751,0.434190,0.961806,0.824486,0.503468,0.554964
6,recon_statistics,52,completed,0.237397,0.558008,0.979167,0.413667,0.666826,0.557001
7,recon_correlation,52,completed,0.179902,0.467254,0.972222,0.613102,0.495181,0.558534
8,full,62,completed,0.241594,0.554637,0.975694,0.395542,0.678732,0.557872
9,recon_only,62,completed,0.163588,0.395080,0.958333,0.732629,0.479722,0.557023


## Cross-domain transfer and few-shot adaptation

For every direction and target-data fraction, the notebook compares:

- training the target model from scratch;
- initializing the shared temporal blocks from the source domain.

The same target training windows, validation data, scenario grid, and threshold protocol are used for the pair.


In [5]:
if RUN_TRANSFER:
    transfer_results = run_cross_domain_transfer_suite(
        config["paths"]["benchmarks"] / "gas_controlled.npz",
        config["paths"]["benchmarks"] / "skab_controlled.npz",
        config=config,
        seeds=SEEDS,
        fractions=TRANSFER_FRACTIONS,
        directions=TRANSFER_DIRECTIONS,
        force=FORCE_RERUN,
    )
    display(transfer_results[[column for column in [
        "direction", "initialization", "requested_target_fraction", "actual_target_fraction",
        "seed", "status", "f1", "auprc", "event_recall", "false_positive_rate"
    ] if column in transfer_results.columns]])


LCAD-AE epoch 1/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 2/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 3/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 4/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 5/30:   0%|          | 0/4 [00:00<?, ?it/s]

...

LCAD-AE epoch 27/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 28/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 29/30:   0%|          | 0/4 [00:00<?, ?it/s]

LCAD-AE epoch 30/30:   0%|          | 0/4 [00:00<?, ?it/s]

,direction,initialization,requested_target_fraction,actual_target_fraction,seed,status,f1,auprc,event_recall,false_positive_rate
0,gas_to_skab,scratch,0.01,0.010691,42,completed,0.384199,0.440607,0.607639,0.014147
1,gas_to_skab,transfer,0.01,0.010691,42,completed,0.372870,0.431837,0.663194,0.029416
2,gas_to_skab,scratch,0.05,0.050606,42,completed,0.380789,0.437747,0.597222,0.014147
3,gas_to_skab,transfer,0.05,0.050606,42,completed,0.394103,0.433720,0.652778,0.018558
4,gas_to_skab,scratch,0.10,0.100499,42,completed,0.453484,0.500299,0.680556,0.018863
...,...,...,...,...,...,...,...,...,...,...
95,skab_to_gas,transfer,0.10,0.101205,82,completed,0.088740,0.171111,0.281250,0.099387
96,skab_to_gas,scratch,0.25,0.250602,82,completed,0.105837,0.173978,0.368056,0.095718
97,skab_to_gas,transfer,0.25,0.250602,82,completed,0.099559,0.175179,0.315972,0.095718
98,skab_to_gas,scratch,1.00,1.000000,82,completed,0.167812,0.194186,0.572917,0.133344


## Proposed-model audit files


In [6]:
for family in ["proposed", "transfer"]:
    result_dir = manuscript_result_root(config) / family
    for file_name in ["runs.csv", "fault_summary.csv", "threshold_sensitivity.csv"]:
        path = result_dir / file_name
        if path.exists():
            print(f"\n{family}/{file_name}")
            display(pd.read_csv(path).tail(20))



proposed/runs.csv


,timestamp_utc,experiment_key,run_id,dataset,model,variant,seed,family,evaluation_set,experiment_version,...,validation_score_median,validation_score_p99,native_precision,native_recall,native_f1,native_auroc,native_auprc,native_event_recall,native_false_positive_rate,native_sequence_count
20,2026-07-29T01:28:48.011297+00:00,5e61b182fc3895c8c752,skab_LCAD_AE_full_42_5e61b182,skab,LCAD_AE,full,42,proposed,scenario_grid,manuscript_v2,...,0.057890,0.096921,0.388326,0.999455,0.557721,0.847919,0.803732,1.0,0.988476,34.0
21,2026-07-29T01:29:35.311302+00:00,2068de53a57f400f3929,skab_LCAD_AE_recon_only_42_2068de53,skab,LCAD_AE,recon_only,42,proposed,scenario_grid,manuscript_v2,...,0.009864,0.014966,0.385419,1.000000,0.554965,0.845323,0.791145,1.0,0.999140,34.0
22,2026-07-29T01:30:22.266867+00:00,4f76a16a7597e009217a,skab_LCAD_AE_recon_statistics_42_4f76a16a,skab,LCAD_AE,recon_statistics,42,proposed,scenario_grid,manuscript_v2,...,0.037574,0.075684,0.387962,1.000000,0.557474,0.845047,0.796566,1.0,0.990024,34.0
23,2026-07-29T01:31:11.106817+00:00,7984531c9efbc9af3b31,skab_LCAD_AE_recon_correlation_42_7984531c,skab,LCAD_AE,recon_correlation,42,proposed,scenario_grid,manuscript_v2,...,0.027900,0.039129,0.389577,1.000000,0.559136,0.853059,0.803761,1.0,0.983832,34.0
24,2026-07-29T01:31:58.275480+00:00,76dae21aa7030c23a9ba,skab_LCAD_AE_full_52_76dae21a,skab,LCAD_AE,full,52,proposed,scenario_grid,manuscript_v2,...,0.068361,0.112456,0.388549,1.000000,0.558034,0.854350,0.810247,1.0,0.988132,34.0
25,2026-07-29T01:32:46.321942+00:00,999102f2eb99c2bcf25c,skab_LCAD_AE_recon_only_52_999102f2,skab,LCAD_AE,recon_only,52,proposed,scenario_grid,manuscript_v2,...,0.014716,0.023484,0.385418,1.000000,0.554964,0.850763,0.797866,1.0,0.999140,34.0
26,2026-07-29T01:33:35.615085+00:00,f073a6f45d2894497607,skab_LCAD_AE_recon_statistics_52_f073a6f4,skab,LCAD_AE,recon_statistics,52,proposed,scenario_grid,manuscript_v2,...,0.046741,0.087974,0.387484,1.000000,0.557001,0.857397,0.814267,1.0,0.991744,34.0
27,2026-07-29T01:34:23.934289+00:00,cc8e932dcfc8ca334cac,skab_LCAD_AE_recon_correlation_52_cc8e932d,skab,LCAD_AE,recon_correlation,52,proposed,scenario_grid,manuscript_v2,...,0.035059,0.047635,0.388926,1.000000,0.558534,0.840281,0.787268,1.0,0.985724,34.0
28,2026-07-29T01:35:11.643339+00:00,82d163aa60865c8f583b,skab_LCAD_AE_full_62_82d163aa,skab,LCAD_AE,full,62,proposed,scenario_grid,manuscript_v2,...,0.061259,0.102044,0.388376,1.000000,0.557872,0.862826,0.820064,1.0,0.988648,34.0
29,2026-07-29T01:35:58.553376+00:00,031173101ecdd7810ba7,skab_LCAD_AE_recon_only_62_03117310,skab,LCAD_AE,recon_only,62,proposed,scenario_grid,manuscript_v2,...,0.011941,0.022566,0.387435,0.999728,0.557023,0.883699,0.852072,1.0,0.990884,34.0



proposed/fault_summary.csv


,fault_type,precision,recall,f1,auroc,auprc,event_recall,mean_detection_delay_windows,false_positive_rate,channel_auprc,...,experiment_key,run_id,dataset,model,variant,seed,family,experiment_version,config_fingerprint,scenario_profile
220,proportional,0.083259,0.423823,0.133120,0.541379,0.143474,0.812500,5.615385,0.382270,0.391184,...,07cf937b061d2a3b646f,skab_LCAD_AE_full_82_07cf937b,skab,LCAD_AE,full,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
221,stuck,0.154164,0.759087,0.248435,0.790372,0.541265,1.000000,3.312500,0.377849,0.752437,...,07cf937b061d2a3b646f,skab_LCAD_AE_full_82_07cf937b,skab,LCAD_AE,full,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
222,additive,0.104767,0.946030,0.183395,0.844577,0.574183,0.979167,0.361702,0.717824,0.645345,...,cb2d66099e57efef38bf,skab_LCAD_AE_recon_only_82_cb2d6609,skab,LCAD_AE,recon_only,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
223,common_mode,0.106603,0.983160,0.187195,0.930777,0.734806,1.000000,0.229167,0.726047,0.804772,...,cb2d66099e57efef38bf,skab_LCAD_AE_recon_only_82_cb2d6609,skab,LCAD_AE,recon_only,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
224,gradual,0.094525,0.862774,0.165549,0.713003,0.354508,1.000000,3.562500,0.721968,0.505334,...,cb2d66099e57efef38bf,skab_LCAD_AE_recon_only_82_cb2d6609,skab,LCAD_AE,recon_only,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
225,noise,0.101789,0.956212,0.179281,0.880933,0.616295,1.000000,0.750000,0.729049,0.725585,...,cb2d66099e57efef38bf,skab_LCAD_AE_recon_only_82_cb2d6609,skab,LCAD_AE,recon_only,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
226,proportional,0.076723,0.691592,0.133765,0.430424,0.075966,0.916667,3.795455,0.722966,0.326818,...,cb2d66099e57efef38bf,skab_LCAD_AE_recon_only_82_cb2d6609,skab,LCAD_AE,recon_only,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
227,stuck,0.078473,0.658675,0.136027,0.408778,0.080818,0.854167,1.853659,0.725998,0.341463,...,cb2d66099e57efef38bf,skab_LCAD_AE_recon_only_82_cb2d6609,skab,LCAD_AE,recon_only,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
228,additive,0.167623,0.905818,0.273565,0.888662,0.622993,1.000000,0.687500,0.378703,0.815384,...,a4d1f6db16ac0ae40304,skab_LCAD_AE_recon_statistics_82_a4d1f6db,skab,LCAD_AE,recon_statistics,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript
229,common_mode,0.179586,0.986065,0.294395,0.960812,0.799262,1.000000,0.125000,0.382919,0.913644,...,a4d1f6db16ac0ae40304,skab_LCAD_AE_recon_statistics_82_a4d1f6db,skab,LCAD_AE,recon_statistics,82,proposed,manuscript_v2,a520efd426e55d648ca4,manuscript



proposed/threshold_sensitivity.csv


...

,timestamp_utc,experiment_key,run_id,dataset,model,variant,seed,family,evaluation_set,experiment_version,...,threshold_method,validation_score_median,validation_score_p99,direction,source_domain,target_domain,requested_target_fraction,initialization,actual_target_fraction,target_train_windows
80,2026-07-29T02:38:34.076714+00:00,07356c61c9687ae78824,gas_to_skab_scratch_0.01_82_07356c61,skab,LCAD_AE_scratch,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,2.255883,2.582337,gas_to_skab,gas_sensor,skab,0.01,scratch,0.010691,15.0
81,2026-07-29T02:39:13.826444+00:00,ae59d50044522fb4d41e,gas_to_skab_transfer_0.01_82_ae59d500,skab,LCAD_AE_transfer,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,2.199819,2.541584,gas_to_skab,gas_sensor,skab,0.01,transfer,0.010691,15.0
82,2026-07-29T02:39:54.610756+00:00,109ace8ee817f3ca948a,gas_to_skab_scratch_0.05_82_109ace8e,skab,LCAD_AE_scratch,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,2.189234,2.507471,gas_to_skab,gas_sensor,skab,0.05,scratch,0.050606,71.0
83,2026-07-29T02:40:35.692281+00:00,d060a6e1c310acb5f42b,gas_to_skab_transfer_0.05_82_d060a6e1,skab,LCAD_AE_transfer,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,2.185740,2.530387,gas_to_skab,gas_sensor,skab,0.05,transfer,0.050606,71.0
84,2026-07-29T02:41:17.039229+00:00,c0b78f36da589086ab02,gas_to_skab_scratch_0.1_82_c0b78f36,skab,LCAD_AE_scratch,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,1.252296,1.503663,gas_to_skab,gas_sensor,skab,0.10,scratch,0.100499,141.0
85,2026-07-29T02:41:56.723073+00:00,eda75b4cfb845bec8422,gas_to_skab_transfer_0.1_82_eda75b4c,skab,LCAD_AE_transfer,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,1.229499,1.529271,gas_to_skab,gas_sensor,skab,0.10,transfer,0.100499,141.0
86,2026-07-29T02:42:37.107363+00:00,ec738384fedeea87b5ea,gas_to_skab_scratch_0.25_82_ec738384,skab,LCAD_AE_scratch,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,0.707531,0.873758,gas_to_skab,gas_sensor,skab,0.25,scratch,0.250178,351.0
87,2026-07-29T02:43:17.786545+00:00,80643ecc24b3087d221d,gas_to_skab_transfer_0.25_82_80643ecc,skab,LCAD_AE_transfer,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,0.629341,0.815879,gas_to_skab,gas_sensor,skab,0.25,transfer,0.250178,351.0
88,2026-07-29T02:44:03.873616+00:00,b71f212720beb1ff0a19,gas_to_skab_scratch_1_82_b71f2127,skab,LCAD_AE_scratch,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,0.066813,0.108915,gas_to_skab,gas_sensor,skab,1.00,scratch,1.000000,1403.0
89,2026-07-29T02:44:50.174996+00:00,211d9ba17aae49834278,gas_to_skab_transfer_1_82_211d9ba1,skab,LCAD_AE_transfer,full,82,transfer,scenario_grid,manuscript_v2,...,quantile,0.058942,0.099355,gas_to_skab,gas_sensor,skab,1.00,transfer,1.000000,1403.0



transfer/fault_summary.csv


,fault_type,precision,recall,f1,auroc,auprc,event_recall,mean_detection_delay_windows,false_positive_rate,channel_auprc,...,family,experiment_version,config_fingerprint,scenario_profile,direction,source_domain,target_domain,requested_target_fraction,actual_target_fraction,initialization
580,proportional,0.016755,0.009846,0.012042,0.461702,0.125536,0.166667,28.250000,0.095770,0.289160,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,scratch
581,stuck,0.305519,0.372390,0.332479,0.701138,0.274323,0.854167,9.048780,0.095765,0.694059,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,scratch
582,additive,0.077903,0.084402,0.077243,0.573172,0.170603,0.333333,14.375000,0.095767,0.467816,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
583,common_mode,0.120571,0.149829,0.131108,0.597374,0.191923,0.354167,9.882353,0.095761,0.646301,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
584,gradual,0.020139,0.013526,0.015749,0.457093,0.132661,0.125000,36.333333,0.095732,0.320478,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
585,noise,0.041540,0.036487,0.037745,0.552038,0.154255,0.145833,20.571429,0.095513,0.441705,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
586,proportional,0.007059,0.004282,0.005215,0.465533,0.125644,0.083333,32.250000,0.095770,0.286455,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
587,stuck,0.303505,0.369657,0.330295,0.702989,0.275988,0.854167,9.219512,0.095765,0.692588,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
588,additive,0.152454,0.236444,0.175931,0.677895,0.201328,0.562500,7.444444,0.130585,0.616858,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,1.00,1.000000,scratch
589,common_mode,0.186771,0.323007,0.227682,0.738022,0.229611,0.687500,13.939394,0.136476,0.696100,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,1.00,1.000000,scratch



transfer/threshold_sensitivity.csv


,threshold_name,threshold,precision,recall,f1,auroc,auprc,event_recall,mean_detection_delay_windows,false_positive_rate,...,family,experiment_version,config_fingerprint,scenario_profile,direction,source_domain,target_domain,requested_target_fraction,actual_target_fraction,initialization
680,quantile_0.975,6.555131,0.102305,0.118940,0.107091,0.558033,0.175179,0.375000,14.953704,0.099387,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
681,quantile_0.99,6.781921,0.095119,0.109697,0.099559,0.558033,0.175179,0.315972,13.923077,0.095718,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
682,quantile_0.995,6.804649,0.094764,0.109348,0.099216,0.558033,0.175179,0.312500,13.955556,0.095718,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
683,mad_4,7.866539,0.072439,0.081706,0.075440,0.558033,0.175179,0.211806,12.803279,0.092035,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
684,mad_6,9.855384,0.052510,0.055909,0.053655,0.558033,0.175179,0.138889,11.900000,0.073634,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
685,mad_8,11.844228,0.046665,0.045446,0.045319,0.558033,0.175179,0.111111,12.625000,0.066282,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,0.25,0.250602,transfer
686,quantile_0.95,3.290043,0.163024,0.255016,0.190913,0.631263,0.194186,0.652778,12.085106,0.140461,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,1.00,1.000000,scratch
687,quantile_0.975,3.450183,0.157307,0.239515,0.182183,0.631263,0.194186,0.631944,12.609890,0.136910,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,1.00,1.000000,scratch
688,quantile_0.99,3.680258,0.147067,0.216976,0.167812,0.631263,0.194186,0.572917,12.757576,0.133344,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,1.00,1.000000,scratch
689,quantile_0.995,3.763935,0.142654,0.206626,0.161513,0.631263,0.194186,0.545139,13.579618,0.130394,...,transfer,manuscript_v2,a520efd426e55d648ca4,manuscript,skab_to_gas,skab,gas_sensor,1.00,1.000000,scratch


## Lightweight-model evidence


In [7]:
proposed_runs = pd.read_csv(manuscript_result_root(config) / "proposed" / "runs.csv")
full = proposed_runs[(proposed_runs["status"] == "completed") & (proposed_runs["variant"] == "full")]
lightweight_summary = full.groupby("dataset", as_index=False).agg(
    parameters=("parameter_count", "mean"),
    model_size_mb=("model_size_mb", "mean"),
    inference_ms_per_window=("inference_ms_per_window", "mean"),
    f1=("f1", "mean"),
    auprc=("auprc", "mean"),
)
display(lightweight_summary)


,dataset,parameters,model_size_mb,inference_ms_per_window,f1,auprc
0,gas_sensor,21792.0,0.092305,0.052036,0.156945,0.189601
1,skab,6072.0,0.032308,0.025839,0.246348,0.552177
